# A little extra!

## New addition to Week 1

### The Unreasonable Effectiveness of the Agent Loop

# What is an Agent?

## Three competing definitions

1. AI systems that can do work for you independently - Sam Altman

2. A system in which an LLM controls the workflow - Anthropic

3. An LLM agent runs tools in a loop to achieve a goal

## The third one is the new, emerging definition

But what does it mean?

Let's make it real.

In [1]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [2]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [3]:
openai = OpenAI()

In [4]:
# Some lists!

todos = []
completed = []

In [5]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [6]:
get_todo_report()

''

In [7]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [8]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [9]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [10]:
mark_complete(1, "bought")

bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [11]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [12]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [13]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [14]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [15]:
def loop(messages):
    done = False
    while not done:
        print (messages)
        response = openai.chat.completions.create(model="gpt-5.2", messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [20]:
tools

[{'type': 'function',
  'function': {'name': 'create_todos',
   'description': 'Add new todos from a list of descriptions and return the full list',
   'parameters': {'type': 'object',
    'properties': {'descriptions': {'type': 'array',
      'items': {'type': 'string'},
      'title': 'Descriptions'}},
    'required': ['descriptions'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'mark_complete',
   'description': 'Mark complete the todo at the given position (starting from 1) and return the full list',
   'parameters': {'properties': {'index': {'description': 'The 1-based index of the todo to mark as complete',
      'title': 'Index',
      'type': 'integer'},
     'completion_notes': {'description': 'Notes about how you completed the todo in rich console markup',
      'title': 'Completion Notes',
      'type': 'string'}},
    'required': ['index', 'completion_notes'],
    'type': 'object',
    'additionalProperties': False}}}]

In [18]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A boat starts to cross a river from east to west which has flows from north to south with 5mph. If the speed of the boat is 10mph, how long it would take for the boat to cross the river. Also what would be the initial angle to start to reach at the oppositie direct location (think like strait line)?

"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [19]:
todos, completed = [], []
loop(messages)

[{'role': 'system', 'content': "\nYou are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.\nNow use the todo list tools, create a plan, carry out the steps, and reply with the solution.\nIf any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.\nProvide your solution in Rich console markup without code blocks.\nDo not ask the user questions or clarification; respond only with the answer after using your tools.\n"}, {'role': 'user', 'content': '"\nA boat starts to cross a river from east to west which has flows from north to south with 5mph. If the speed of the boat is 10mph, how long it would take for the boat to cross the river. Also what would be the initial angle to start to reach at the oppositie direct location (think like strait line)?\n\n'}]


Todo #1: Identify knowns/unknowns and note missing quantity (river width).
Todo #2: Express crossing time as function of river width using boat’s effective westward component.
Todo #3: Compute required heading angle to land directly opposite (cancel southward current) and check feasibility.
Todo #4: Provide final answers with a reasonable width estimate example since width not given.

[{'role': 'system', 'content': "\nYou are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.\nNow use the todo list tools, create a plan, carry out the steps, and reply with the solution.\nIf any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.\nProvide your solution in Rich console markup without code blocks.\nDo not ask the user questions or clarification; respond only with the answer after using your tools.\n"}, {'role': 'user', 'content': '"\nA boat starts to cross a river from east to west which has flows from north to south with 5mph. If the speed of the boat is 10mph, how long it would take for the boat to cross the river. Also what would be the initial angle to start to reach at the oppositie direct location (think like strait line)?\n\n'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[

Given: current 
- River flows north → south at 5 mph.
- Boat speed relative to water: 10 mph.
Unknown needed for time: river width W (east–west distance). Not provided, so time can only be expressed in terms 
of W or with an assumed W.

Todo #1: Identify knowns/unknowns and note missing quantity (river width).
Todo #2: Express crossing time as function of river width using boat’s effective westward component.
Todo #3: Compute required heading angle to land directly opposite (cancel southward current) and check feasibility.
Todo #4: Provide final answers with a reasonable width estimate example since width not given.

...

Todo #1: Identify knowns/unknowns and note missing quantity (river width).
Todo #2: Express crossing time as function of river width using boat’s effective westward component.
Todo #3: Compute required heading angle to land directly opposite (cancel southward current) and check feasibility.
Todo #4: Provide final answers with a reasonable width estimate example since width not given.

Since W not given, provide (1) formula in terms of W, and (2) one reasonable example width.
Example assumption: W = 0.5 mile (a moderate river). Then t = 0.5/8.66 hr ≈ 0.0577 hr ≈ 3.46 minutes.

Todo #1: Identify knowns/unknowns and note missing quantity (river width).
Todo #2: Express crossing time as function of river width using boat’s effective westward component.
Todo #3: Compute required heading angle to land directly opposite (cancel southward current) and check feasibility.
Todo #4: Provide final answers with a reasonable width estimate example since width not given.

[{'role': 'system', 'content': "\nYou are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.\nNow use the todo list tools, create a plan, carry out the steps, and reply with the solution.\nIf any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.\nProvide your solution in Rich console markup without code blocks.\nDo not ask the user questions or clarification; respond only with the answer after using your tools.\n"}, {'role': 'user', 'content': '"\nA boat starts to cross a river from east to west which has flows from north to south with 5mph. If the speed of the boat is 10mph, how long it would take for the boat to cross the river. Also what would be the initial angle to start to reach at the oppositie direct location (think like strait line)?\n\n'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[

Given  
• Current: 5 mph north → south  
• Boat speed in still water: 10 mph  
• River width: W miles (not provided)

1) Heading to arrive directly opposite  
To land straight across (no net north/south drift), the boat must aim north of west so its northward component 
cancels the 5 mph southward current:  
• 10·sinθ = 5 ⟹ sinθ = 0.5 ⟹ θ = 30°  
So you should start at 30° north of west (equivalently, 60° west of north).

2) Time to cross  
With that heading, the westward (across-river) speed is:  
• 10·cos30° = √(10²−5²) = √75 ≈ 8.66 mph  

So the crossing time is  
t = W / 8.66 hours  (with W in miles)

Example (since W isn’t given): if W = 0.5 mile, then  
t = 0.5 / 8.66 hr ≈ 0.0577 hr ≈ 3.46 minutes.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>